# X2P Rabi 幅度校准

该流程用两个连续 `X2P` 扫描 active XY2 setting 的幅度。实验只生成候选值；确认前不会修改当前配置。

In [ ]:
from uuid import uuid4

from sqvm.calibration import (
    apply_calibration_candidates_to_current_configuration,
    run_rabi,
)

print('SQVM Rabi 用户接口导入成功')

## 扫描参数

设置目标、幅度范围和步进，单位均为 GHz。范围必须从零开始，端点会被包含。其余执行参数使用 API 默认值。

In [ ]:
TARGET = 'Q1'
AMPLITUDE_RANGE_GHZ = (0.0, 0.03)
AMPLITUDE_STEP_GHZ = 0.002
OPERATION_ID = str(uuid4())
RUN_EXPERIMENT = True
UPDATE_CANDIDATE = False

In [ ]:
def report_progress(event):
    action = '开始' if event['event'] == 'circuit_started' else '完成'
    current = event['completed'] + 1 if event['event'] == 'circuit_started' else event['completed']
    print(f"{action} {current}/{event['total']}: {event['circuit_id']}")

if RUN_EXPERIMENT:
    result = run_rabi(
        target=TARGET,
        amplitude_range_GHz=AMPLITUDE_RANGE_GHZ,
        amplitude_step_GHz=AMPLITUDE_STEP_GHZ,
        operation_id=OPERATION_ID,
        progress_callback=report_progress,
    )
    target_data = result.data.get(TARGET)
    if target_data is None:
        raise RuntimeError(f'运行结果不包含目标 {TARGET} 的 Rabi 数据')
    print(f'运行 ID: {result.run_id}')
    print(f'结果目录: {result.root}')
    print(f'amplitude_GHz: {list(target_data["amplitude_GHz"])}')
    print(f'P1: {list(target_data["P1"])}')
    candidate = result.candidates.get(TARGET)
    if candidate is None:
        print(f'目标 {TARGET} 未生成校准候选；请检查分析和质量门。')
    elif candidate.get('recommendation_eligible') is not True:
        print(f"候选不可应用：{candidate.get('reason') or 'analysis policy 或质量门未通过'}")
    else:
        print(f'候选: {candidate}')
else:
    print('参数已设置。将 RUN_EXPERIMENT 改为 True 后重新运行本单元格。')

## 查看并确认候选

检查首个 Rabi 峰、拟合质量和候选幅度。确认无误后将 `UPDATE_CANDIDATE` 改为 `True`，再运行下方单元格。

In [ ]:
if UPDATE_CANDIDATE:
    if not RUN_EXPERIMENT or 'result' not in globals():
        raise RuntimeError('请先运行实验并检查候选幅度')
    candidate = result.candidates.get(TARGET)
    if candidate is None:
        raise RuntimeError(f'目标 {TARGET} 未生成校准候选；不会写入当前配置')
    if candidate.get('recommendation_eligible') is not True:
        reason = candidate.get('reason') or 'analysis policy 或质量门未通过'
        raise RuntimeError(f'候选不可应用：{reason}；不会写入当前配置')
    candidate_id = candidate.get('candidate_id')
    if not isinstance(candidate_id, str) or not candidate_id:
        raise RuntimeError(f'目标 {TARGET} 的候选 ID 无效；不会写入当前配置')
    update = apply_calibration_candidates_to_current_configuration(
        result,
        confirmation_phrase=f'APPLY CALIBRATION CANDIDATES {result.run_id}',
        candidate_ids=[candidate_id],
        actor_id='project.manager',
    )
    print(f'已更新当前配置: {update.device_id} r{update.current_revision}')
    print(f'已更新参数: {dict(update.applied_values)}')
else:
    print('候选幅度尚未写入当前配置；确认后将 UPDATE_CANDIDATE 改为 True。')

## Web 查看结果

启动 `start_calibration_web.cmd` 后访问 [http://127.0.0.1:8765/#/experiments](http://127.0.0.1:8765/#/experiments)。Web 控制台只查看结果和确认候选，不启动实验。